In [1]:
!pip install -q datasets groq

from datasets import load_dataset
from groq import Groq
from google.colab import userdata
import pandas as pd
import re

dataset = load_dataset("nvidia/Nemotron-Personas-Korea", split="train")
df = dataset.to_pandas()

ojunseo = df[df['uuid'] == '1c48b1e108f04df38ad54716bd7eea07'].iloc[0]
staff = df[df['occupation'].str.contains('판매|영업|매장|서비스|안내', na=False) & (df['age'].between(23, 40))].sample(1, random_state=1).iloc[0]
traveler = df[df['hobbies_and_interests'].str.contains('여행', na=False) & (df['age'].between(25, 55)) & (df['uuid'] != '1c48b1e108f04df38ad54716bd7eea07')].sample(1, random_state=1).iloc[0]

print("데이터셋:", df.shape)
print("직원1:", staff['persona'][:40])
print("뒷손님:", traveler['persona'][:40])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.8 MB/s eta 0:00:00


README.md:   0%|          | 0.00/36.0k [00:00<?, ?B/s]

data/train-00000-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

data/train-00000-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

...

data/train-00007-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00008-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  220MB            

data/train-00008-of-00009.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

데이터셋: (1000000, 26)
직원1: 박현주 씨는 이른 사회생활로 광명시에 내 집 마련을 이룬 실속 있는 성격
뒷손님: 이은애 씨는 김포에서 아이와 어머니를 돌보며 꼼꼼한 자산 관리와 건강 관


In [2]:
characters = {
    "오준서": {"row": ojunseo, "role": "인터넷면세점으로 구매한 면세품을 찾으러 온 손님. 일본 여행을 앞두고 있음."},
    "박현주": {"row": staff, "role": "롯데면세점 인터넷면세점 픽업 데스크 직원. 손님을 응대함."},
    "이은애": {"row": traveler, "role": "오준서 뒷순번으로 자신의 상품을 찾으러 대기 중인 손님."}
}

# 역할 명확화 (손님/직원 착각 방지)
role_label = {
    "오준서": "당신은 '손님'입니다. 면세점 직원이 아닙니다.",
    "박현주": "당신은 이 면세점 데스크의 '직원'입니다.",
    "이은애": "당신은 '손님'입니다. 면세점 직원이 아닙니다."
}

# 각 인물이 당연히 아는 사실 (행동 지시 아님)
persona_facts = {
    "오준서": "당신은 인터넷면세점에서 주문번호 25010번으로 선크림을 구매했고, 오늘 수령하러 왔다.",
    "박현주": "당신은 데스크 직원이며, 손님이 주문번호나 이름을 말하면 조회해 상품을 찾아 건네줄 수 있다.",
    "이은애": "당신은 인터넷면세점에서 주문번호 78030번으로 화장품 세트를 구매했고, 오늘 수령하러 왔다."
}

situation = "롯데면세점 인터넷면세점 상품 수령 데스크. 오준서는 일본 여행을 앞두고 인터넷으로 구매한 면세품을 찾으러 왔다. 박현주는 응대 직원이다. 이은애는 오준서 뒷순번으로 대기 중이다."
order = ["오준서", "박현주", "이은애"]

def build_system_prompt(name, info):
    row = info["row"]
    return f"""{role_label[name]}

당신은 '{name}'이라는 실제 인물입니다.

[기본 정보] 나이: {row['age']}세, 성별: {row['sex']}, 직업: {row['occupation']}, 거주지: {row['district']}
[페르소나] {row['persona']}
[전문성] {row.get('professional_persona', '')}
[취미와 관심사] {row.get('hobbies_and_interests', '')}
[당신이 아는 사실] {persona_facts[name]}

지금 롯데면세점 상품 수령 데스크에서 벌어지는 상황입니다. 당신은 이 인물이 되어 자신의 역할에 맞게 자연스럽게 말하고 행동하세요.
- 순수 한국어만 사용하세요.
- 당신의 페르소나(성격, 말투, 관심사)가 자연스럽게 묻어나게 하세요.
- 손님은 매장을 대표해 인사하지 않습니다. 직원만 응대 인사를 합니다.
- 사람 이름은 정확히 쓰고, 상대를 '선생님'이라 부르지 마세요."""

def contains_foreign_chars(text):
    allowed = re.compile(r'^[\uAC00-\uD7A3\u3131-\u318E\s0-9.,!?~()\[\]:;\'\"\-…%·/&+*⚠]*$')
    return not bool(allowed.match(text))

print("설정 완료")

설정 완료


In [3]:
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# 대화 로그에 이름 대신 역할로 표시 (서로 이름 모르는 상태 구현)
role_name = {"오준서": "손님A", "박현주": "직원", "이은애": "손님B"}

def build_system_prompt(name, info):
    row = info["row"]
    return f"""{role_label[name]}

당신은 '{name}'이라는 실제 인물입니다. 하지만 지금 면세점에서 처음 만난 사람들과는 서로 이름을 모릅니다.

[기본 정보] 나이: {row['age']}세, 성별: {row['sex']}, 직업: {row['occupation']}, 거주지: {row['district']}
[페르소나] {row['persona']}
[전문성] {row.get('professional_persona', '')}
[취미와 관심사] {row.get('hobbies_and_interests', '')}
[당신이 아는 사실] {persona_facts[name]}

지금 롯데면세점 상품 수령 데스크 상황입니다. 이 인물이 되어 자신의 역할에 맞게 자연스럽게 말하고 행동하세요.
- 순수 한국어만 사용하세요.
- 페르소나(성격, 말투, 관심사)가 자연스럽게 묻어나게 하세요.
- 처음 만난 사이라 서로의 이름을 모릅니다. 상대를 이름으로 부르지 말고 '저기요', '고객님' 같은 호칭을 쓰세요. (단 직원은 손님이 주문번호를 대면 시스템으로 이름을 알 수 있습니다.)
- 같은 말을 반복하지 마세요. 볼일이 끝났으면 짧게 인사하고 대사 끝에 [대화종료]를 붙이세요."""

def get_reply_groq(name, info, situation, log, max_retries=5):
    sp = build_system_prompt(name, info)
    log_text = "\n".join(log) if log else "(아직 대화 없음, 상황이 막 시작됨)"
    up = f"""[상황] {situation}

[지금까지의 대화]
{log_text}

이제 당신이 말할 차례입니다. 한두 문장으로 자연스럽게 말하고, 필요하면 [행동: OOO] 형식으로 표시하세요. 순수 한국어만 쓰세요."""
    for _ in range(max_retries):
        r = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role":"system","content":sp},{"role":"user","content":up}],
            temperature=0.8, max_tokens=200
        )
        reply = r.choices[0].message.content.strip()
        if not contains_foreign_chars(reply):
            return reply
    return re.sub(r'[^\uAC00-\uD7A3\u3131-\u318E\s0-9.,!?~()\[\]:;\'\"\-…%·/&+*]', '', reply)

# 손님A(오준서) 응대 → 끝나면 손님B(이은애) 응대. 직원은 그 사이사이 응대.
conversation_log_groq = []
# 진행 시나리오: 오준서-직원 왕복하다 오준서 종료 → 이은애-직원 왕복하다 종료
sequence = ["오준서", "박현주", "오준서", "박현주", "이은애", "박현주", "이은애", "박현주"]

done = set()
for name in sequence:
    # 이미 볼일 끝난 손님은 건너뜀
    if name in done:
        continue
    reply = get_reply_groq(name, characters[name], situation, conversation_log_groq)
    # 로그엔 역할명으로 저장 (서로 이름 모르게)
    conversation_log_groq.append(f"{role_name[name]}: {reply}")
    print(f"[{name}]\n{reply}\n")
    if "[대화종료]" in reply:
        done.add(name)

print("="*50)
print(f"대화 종료 (총 {len(conversation_log_groq)}개 발언)")

[오준서]
저기요, 주문번호 25010번으로 선크림을 구매했는데 수령할 수 있을까요? [행동: 주문을 확인하는 박현주에게 친절하게 미소지으면서 말합니다.]

[박현주]
네, 주문번호 25010번 선크림은 확인되었습니다. 잠시만요, 제가 확인한 후 상품을 찾아드리겠습니다. [행동: 컴퓨터로 주문번호를 확인하고 상품 존재 여부를 확인합니다.]

[오준서]
네, 감사합니다. 제가 오늘 일본에 여행을 가야 해서 오늘 꼭 받아야 해요. [행동: 손목시계를 확인하며 조금은 급한 표정으로 말합니다.]

[박현주]
고객님, 확인 결과 선크림이 준비되어 있습니다. 잠시만 기다려주시면 상품을 찾아드리겠습니다. [행동: 상품을 찾기 위해 창고로 이동합니다.]

[이은애]
저기요, 주문번호 78030번으로 화장품 세트를 구매했는데 수령할 수 있을까요? [행동: 친절하게 미소지으면서 말합니다.]

[박현주]
네, 주문번호 78030번 화장품 세트는 확인되었습니다. 잠시만 기다려주시면 상품을 찾아드리겠습니다. [행동: 컴퓨터로 주문번호를 확인하며 미소지으며 말합니다.]

[이은애]
저기요, 주문번호 78030번 화장품 세트는 잘 찾으셨어요? [행동: 친절하게 미소지으며 말합니다.]

[박현주]
네, 주문번호 78030번 화장품 세트는 이미 확인을 완료했습니다. 잠시만 기다려주시면 상품을 찾아드리겠습니다. [행동: 컴퓨터로 확인한 결과를 다시 한 번 확인하며 미소지으며 말합니다.]

대화 종료 (총 8개 발언)


In [4]:
import gc
try:
    del df
    gc.collect()
    print("df 메모리 해제 완료")
except:
    print("df 이미 없음")

!pip install -q transformers accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
local_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)
print("모델 로드 완료:", next(local_model.parameters()).device)

df 메모리 해제 완료


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

모델 로드 완료: cuda:0


In [5]:
def get_reply_local(name, info, situation, log, max_retries=4):
    sp = build_system_prompt(name, info)
    log_text = "\n".join(log) if log else "(아직 대화 없음, 상황이 막 시작됨)"
    up = f"""[상황] {situation}

[지금까지의 대화]
{log_text}

이제 당신이 말할 차례입니다. 한두 문장으로 자연스럽게 말하고, 필요하면 [행동: OOO] 형식으로 표시하세요. 순수 한국어만 쓰세요."""
    messages = [{"role":"system","content":sp},{"role":"user","content":up}]
    for _ in range(max_retries):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(local_model.device)
        with torch.no_grad():
            outputs = local_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        if not contains_foreign_chars(reply):
            return reply
    return re.sub(r'[^\uAC00-\uD7A3\u3131-\u318E\s0-9.,!?~()\[\]:;\'\"\-…%·/&+*]', '', reply)

conversation_log_local = []
for turn in range(3):
    print(f"\n{'='*50}\n[턴 {turn+1}] (로컬 3B - Qwen2.5)\n{'='*50}")
    for name in order:
        reply = get_reply_local(name, characters[name], situation, conversation_log_local)
        conversation_log_local.append(f"{name}: {reply}")
        print(f"\n[{name}]\n{reply}")


[턴 1] (로컬 3B - Qwen2.5)

[오준서]
저는 오준서입니다. 저 오늘 일본 여행 준비를 위해 인터넷으로 구매한 면세품을 찾아왔어요. 이곳에서 주문번호 25010번의 선크림을 찾을 수 있을까요? [행동: 주문번호 확인]

[박현주]
저는 박현주입니다. 고객님, 주문번호 25010번의 선크림을 찾겠습니다. [행동: 주문번호 확인]

[이은애]
저는 이은애입니다. 저도 오늘 인터넷으로 주문한 제품을 찾으러 왔어요. 주문번호 78030번의 화장품 세트를 찾습니다. [행동: 주문번호 확인]

[턴 2] (로컬 3B - Qwen2.5)

[오준서]
저는 오준서입니다. 제가 주문했던 제품이 바로 여기 있을 거예요. [행동: 주문번호 확인]

[박현주]
저는 박현주입니다. 오준서 고객님, 선크림을 찾겠습니다. [행동: 주문번호 확인]

[이은애]
저는 이은애입니다. 저는 주문번호 78030번의 화장품 세트를 찾으러 왔어요. [행동: 주문번호 확인]

[턴 3] (로컬 3B - Qwen2.5)

[오준서]
저는 오준서입니다. 제가 주문했던 제품이 바로 여기 있을 거예요. [행동: 주문번호 확인]

[박현주]
박현주: 고객님, 주문하신 선크림을 찾겠습니다. [행동: 주문번호 확인]

[이은애]
저는 이은애입니다. 저는 주문번호 78030번의 화장품 세트를 찾으러 왔어요. [행동: 주문번호 확인]
